# Custom Tokenizers — Model Training

This notebook documents the training of the BPE (Byte-Pair Encoding) tokenizers for Yoruba, Igbo, Hausa, Pidgin, and the Unified (Naija) model.

## Methodology
1. **Byte-Level BPE (BBPE)**: Operating on raw bytes to ensure zero out-of-vocabulary (`[UNK]`) tokens and perfect whitespace reconstruction.
2. **Diacritic Preservation & Normalization**: Uses Rust-based `NFC` normalizer to ensure diacritics are canonicalized before tokenizing.
3. **Code-Mixed English Support**: Blends a portion of English text (~15% by word count) into the corpus to represent code-mixed sentences efficiently.
4. **Emoji & Symbol Support**: Injects repeated common emojis into the corpus to force the BPE trainer to merge their bytes into single tokens.

In [7]:
# Install required dependencies directly in the active Jupyter kernel environment
%pip install datasets torchcodec

Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
import unicodedata
from datasets import load_dataset, Features, Value
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.normalizers import NFC

/Users/olumideola/Desktop/olaverse-ai/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Production Yoruba Tokenizer (Merged Corpus with Code-Mixed & Emoji Support)

We build a production-grade Yoruba tokenizer by streaming and merging four datasets:
1. **WaxalNLP (`yor_tts`)**: Spoken, colloquial scripts.
2. **Wikipedia (`yo`)**: General-domain encyclopedic texts.
3. **MasakhaNEWS (`yor`)**: Modern news articles and headlines.
4. **Wikipedia (`en`)**: Clean English text (first 3,000 articles) to represent code-mixed structures.
5. **Curated Emojis**: Top 70 emojis injected repeated 100 times to satisfy minimum training frequencies.

In [9]:
def clean_and_normalize(text):
    if not text:
        return ""
    text = unicodedata.normalize("NFC", text)
    text = " ".join(text.split())
    return text

corpus_path = "yoruba_merged_corpus.txt"
print("Streaming and merging datasets...")

with open(corpus_path, "w", encoding="utf-8") as f_out:
    # 1. WaxalNLP
    features = Features({
        'id': Value('string'),
        'speaker_id': Value('string'),
        'text': Value('string'),
        'locale': Value('string'),
        'gender': Value('string'),
        'audio': {'path': Value('string'), 'bytes': Value('binary')}
    })
    waxal = load_dataset('google/WaxalNLP', 'yor_tts', streaming=True, features=features)
    for item in waxal['train']:
        text = clean_and_normalize(item['text'])
        if text:
            f_out.write(text + "\n")
            
    # 2. Wikipedia (Yoruba)
    wiki = load_dataset('wikimedia/wikipedia', '20231101.yo', streaming=True)
    for item in wiki['train']:
        text = clean_and_normalize(item['text'])
        if text:
            f_out.write(text + "\n")
            
    # 3. MasakhaNEWS (Yoruba)
    news = load_dataset('masakhane/masakhanews', 'yor', streaming=True)
    for split in ['train', 'validation', 'test']:
        for item in news[split]:
            text = f"{item.get('headline', '')} {item.get('text', '')}"
            text = clean_and_normalize(text)
            if text:
                f_out.write(text + "\n")

    # 4. Wikipedia (English - first 3000 articles)
    wiki_en = load_dataset('wikimedia/wikipedia', '20231101.en', streaming=True)
    count = 0
    for item in wiki_en['train']:
        text = clean_and_normalize(item['text'])
        if text:
            f_out.write(text + "\n")
            count += 1
            if count >= 3000:
                break
                
    # 5. Inject popular emojis
    emojis_list = [
        "😂", "❤️", "🔥", "😊", "👍", "😭", "😘", "💕", "😍", "✨", 
        "🌟", "🎉", "👏", "🙏", "💪", "🤔", "👀", "✔️", "💯", "🚨", 
        "📌", "📍", "⚠️", "💀", "🚀", "💡", "📱", "💻", "🌍", "🇳🇬", 
        "✈️", "💵", "💰", "🛍️", "💬", "📣", "🔔", "🎵", "🍔", "🍕", 
        "🍻", "🍷", "☕", "🎂", "🎈", "🎁", "🚘", "🛸", "🍿", "🌈", 
        "⭐", "☀️", "❄", "🍀", "🐶", "🐱", "🤖", "🦊", "🦁", "🐯", 
        "⚽", "🏆", "🎮", "🎲", "🎨", "🎬", "🎤", "🎧", "🎸", "🏎️"
    ]
    for emoji in emojis_list:
        f_out.write((emoji + " ") * 100 + "\n")

print("Merged corpus saved to:", corpus_path)

Streaming and merging datasets...


Merged corpus saved to: yoruba_merged_corpus.txt


In [10]:
print("Training Byte-Level BPE tokenizer (vocab_size=50,000)...")
tokenizer = Tokenizer(BPE())
tokenizer.normalizer = NFC()
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(
    special_tokens=["[PAD]", "[CLS]", "[SEP]", "[MASK]"],
    vocab_size=50000,
    min_frequency=2
)
tokenizer.train([corpus_path], trainer)

base_dir = "../../olaverse/models"
save_path = os.path.join(base_dir, "tokenizer_yoruba.json")
os.makedirs(os.path.dirname(save_path), exist_ok=True)
tokenizer.save(save_path)
print("Yoruba tokenizer saved to:", save_path)

if os.path.exists(corpus_path):
    os.remove(corpus_path)

Training Byte-Level BPE tokenizer (vocab_size=50,000)...



Yoruba tokenizer saved to: ../../olaverse/models/tokenizer_yoruba.json


In [11]:
# Verify diacritic preservation and perfect space reconstruction
input_text = "Ẹ kú àbọ̀, ṣé dáadáa ni?"
encoded = tokenizer.encode(input_text)
print(f"Input: '{input_text}'")
print(f"Tokens: {encoded.tokens}")
print(f"Decoded: '{tokenizer.decode(encoded.ids)}'")

# Verify Code-Mixed encoding
mixed_text = "I mean, ko si wahala lórí ọ̀rọ̀ náà."
encoded_mixed = tokenizer.encode(mixed_text)
print(f"\nMixed Input: '{mixed_text}'")
print(f"Mixed Tokens: {encoded_mixed.tokens}")

# Verify Emoji encoding
emoji_text = "😂 ❤️ 🔥 😊 👍"
encoded_emoji = tokenizer.encode(emoji_text)
print(f"\nEmoji Input: '{emoji_text}'")
print(f"Emoji Tokens: {encoded_emoji.tokens}")

Input: 'Ẹ kú àbọ̀, ṣé dáadáa ni?'
Tokens: ['áº¸', 'ĠkÃº', 'ĠÃłbá»į', 'ÌĢ,', 'Ġá¹£Ã©', 'ĠdÃ¡adÃ¡a', 'Ġni', '?']
Decoded: 'Ẹ kú àbọ̀, ṣé dáadáa ni?'

Mixed Input: 'I mean, ko si wahala lórí ọ̀rọ̀ náà.'
Mixed Tokens: ['I', 'Ġmean', ',', 'Ġko', 'Ġsi', 'Ġwahala', 'ĠlÃ³rÃŃ', 'Ġá»į', 'ÌĢ', 'rá»į', 'ÌĢ', 'ĠnÃ¡Ãł', '.']

Emoji Input: '😂 ❤️ 🔥 😊 👍'
Emoji Tokens: ['ðŁ', 'ĺ', 'Ĥ', 'ĠâĿ¤ï¸ı', 'ĠðŁĶ¥', 'ĠðŁĺĬ', 'ĠðŁĳį']
